In [1]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

c:\Users\arote\anaconda3\envs\animal_fixed\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_parquet(
    "../data/processed/summaries_only.parquet"
)

print(df.shape)

(26688, 3)


In [3]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

c:\Users\arote\anaconda3\envs\animal_fixed\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\arote\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2712.36it/s]


In [6]:
index = faiss.read_index(
    "../models/retrieval/faiss_index.bin"
)

print(
    "Vectors:",
    index.ntotal
)

Vectors: 26688


In [7]:
queries = [

    "freedom of speech",

    "right to life",

    "right to privacy",

    "constitutional validity",

    "property dispute",

    "land acquisition compensation",

    "specific performance contract",

    "agent commission",

    "arbitration agreement",

    "consumer protection",

    "industrial dispute",

    "service promotion",

    "murder conviction",

    "bail application",

    "illegal detention",

    "tax assessment",

    "election dispute",

    "freedom of religion",

    "public interest litigation",

    "fundamental rights"

]

In [8]:
def search_cases(
    query,
    top_k=5
):

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    return indices[0], scores[0]

In [9]:
for query in queries:

    print("\n")
    print("="*80)

    print(
        "QUERY:",
        query
    )

    print("="*80)

    indices, scores = search_cases(query)

    for rank, idx in enumerate(indices):

        print(
            rank+1,
            df.iloc[idx]["case_name"]
        )



QUERY: freedom of speech
1 Prabha_Dutt_vs_Union_Of_India_Ors_on_7_November_1981_1
2 Bijoy_Singh_Anr_vs_State_Of_Bihar_on_17_April_2002_1
3 Nazir_Khan_And_Ors_vs_State_Of_Delhi_on_22_August_2003_1
4 Munshi_Singh_Gautam_D_Ors_vs_State_Of_M_P_on_16_November_2004_1
5 Smt_Shakila_Abdul_Gafar_Khan_vs_Vasant_Raghunath_Dhoble_And_Anr_on_8_September_2003_1


QUERY: right to life
1 International_Airport_Authority_Of_vs_K_D_Bali_Another_on_29_March_1988_1
2 Balwant_Singh_vs_Union_Of_India_on_3_May_2023_1
3 Vikram_Deo_Singh_Tomar_vs_State_Of_Bihar_on_2_August_1988_1
4 Prem_Shankar_Shukla_vs_Delhi_Administration_on_29_April_1980_1
5 Justice_Sunanda_Bhandare_Foundation_vs_U_O_I_Anr_on_26_March_2014_1


QUERY: right to privacy
1 Prabha_Dutt_vs_Union_Of_India_Ors_on_7_November_1981_1
2 State_Of_Madhya_Pradesh_vs_Shobharam_And_Ors_on_22_April_1966_1
3 V_N_Shrikhande_vs_Anita_Sena_Fernandes_on_20_October_2010_1
4 Prem_Shankar_Shukla_vs_Delhi_Administration_on_29_April_1980_1
5 M_S_Magma_Fincorp_Ltd_Fo

In [10]:
case_idx = 0

case_text = df.iloc[
    case_idx
]["legal_summary"]

query_embedding = model.encode(
    [case_text],
    normalize_embeddings=True
)

scores, indices = index.search(
    query_embedding,
    10
)

for i, idx in enumerate(indices[0]):

    print(
        i+1,
        df.iloc[idx]["case_name"],
        round(
            float(scores[0][i]),
            4
        )
    )

1 Deb_Sadhan_Roy_vs_State_Of_West_Bengal_on_7_December_1971_1 0.3189
2 Godavari_Parulekar_vs_State_Of_Bombay_And_Others_on_5_December_1952_1 0.3186
3 Sasthi_Chandra_Roy_vs_The_State_Of_West_Bengal_on_24_April_1972_1 0.3143
4 Boppanna_Venkateswaraloo_And_Others_vs_Superintendent_Central_on_24_November_1952_1 0.3114
5 Satya_Deo_Prasad_Gupta_vs_The_State_Of_Bihar_Ors_on_27_November_1974_1 0.3102
6 Dr_Ram_Manohar_Lohia_vs_State_Of_Bihar_And_Others_on_7_September_1965_1 0.3083
7 Abdul_Aziz_vs_The_Distt_Magistrate_Burdwan_Ors_on_11_October_1972_1 0.3078
8 V_C_Mohan_vs_Union_Of_India_Ors_on_1_March_2002_1 0.3043
9 Bidya_Deb_Barma_Etc_vs_District_Magistrate_Tripura_on_6_August_1968_1 0.3006
10 Keshab_Roy_vs_The_State_Of_West_Bengal_on_3_February_1972_1 0.2982
